# 🔬 Transformer 推理视角 — 训练与推理的本质差异

**前置阅读**：建议先读完 `00-llm-inference-primer.ipynb`，理解 token 和自回归生成的基本概念。

**本文目标**：从**推理**视角重新审视 Transformer——不讲 Attention 的完整数学推导（网上已经有很多优秀资料），而是聚焦在**训练和推理的根本差异**，以及这些差异如何催生了后续所有的推理优化技术。

读完这篇你会理解：
- 为什么 Prefill 和 Decode 是两个完全不同的计算阶段
- KV Cache 到底是什么，为什么它是显存瓶颈
- 一次推理请求的显存都花在哪了

## 1. 训练 vs 推理：根本差异

### 训练（Teacher Forcing）

训练时，模型**一步看到全部答案**：

```
输入:  "The cat sat on the"
标签:  "The cat sat on the mat"

前向传播（一次）:
  Position 0: "The"  → 预测 "cat"   ✓/✗
  Position 1: "cat"  → 预测 "sat"   ✓/✗
  Position 2: "sat"  → 预测 "on"    ✓/✗
  Position 3: "on"   → 预测 "the"   ✓/✗
  Position 4: "the"  → 预测 "mat"   ✓/✗

  ← 所有位置并行计算，一次矩阵乘法算出全部位置的 Attention →
```

训练的关键：**Teacher Forcing**——把正确答案喂给模型，所有位置的预测**并行**完成。一次前向传播就能算出整个序列的 loss。

### 推理（Autoregressive）

推理时，模型必须**一步一个脚印**：

```
输入: "The cat sat on the"

Step 1: 输入 "The cat sat on the" → 模型运算 → 输出 "mat"
Step 2: 输入 "The cat sat on the mat" → 模型运算 → 输出 "."
Step 3: 输入 "The cat sat on the mat." → 模型运算 → 输出 <EOS>

← 每次只多生成一个 token，必须跑一次完整的模型前向传播 →
```

| 维度 | 训练 | 推理 |
|------|------|------|
| 输入 | 完整序列（已知答案） | 逐渐增长的序列 |
| 并行度 | 所有位置并行 | 每次只能算一个新位置 |
| 计算次数 | 1 次前向 / 整个序列 | N 次前向 / N 个 token |
| 中间结果 | 丢弃（不缓存） | **必须缓存 KV**，否则要重算 |
| 显存瓶颈 | 优化器状态 + 梯度 + 激活 | 模型权重 + **KV Cache** |
| 计算瓶颈 | 算力（FLOPs） | 小 batch: 显存带宽 / 大 batch: 算力 |

## 2. Attention 的计算视角

先快速回顾 Attention 做了什么（用你最熟悉的视角）：

```python
# 单头 Self-Attention (简化)
def attention(Q, K, V, mask):
    # Q, K, V: [seq_len, d_head] —— 分别从输入 x 乘 W_Q, W_K, W_V 得到
    scores = Q @ K.T / sqrt(d_k)      # [seq_len, seq_len] — 每个位置对其他位置的"关注度"
    scores = softmax(scores + mask)    # mask 防止看到未来 token（causal mask）
    output = scores @ V                # [seq_len, d_head] — 加权聚合
    return output

# 多头 Attention
def multi_head_attention(x, W_Q, W_K, W_V, W_O, n_heads):
    # x: [batch, seq_len, d_model]
    # 对每个头独立做 attention，最后拼接
    per_head = d_model // n_heads
    outputs = []
    for h in range(n_heads):
        q = x @ W_Q[h]   # [batch, seq_len, per_head]
        k = x @ W_K[h]   # [batch, seq_len, per_head]
        v = x @ W_V[h]   # [batch, seq_len, per_head]
        outputs.append(attention(q, k, v, causal_mask))
    return concat(outputs) @ W_O
```

### 关键观察：Q 和 K/V 的不对称性

在自回归推理中，每生成一个新 token：

```
新 token 的 Q:  只看这个新 token 的信息 → 很小 [1, d_head]
历史的 K 和 V:  需要所有历史 token 的信息 → 很大 [seq_len, d_head]
```

**如果每步都重算全部 K、V，Attention 的复杂度是 O(n²)**（n = 序列长度）。这是不可接受的。

解决方案：**缓存 K 和 V**——这就是 KV Cache。

## 3. KV Cache：推理的核心数据结构

### 为什么缓存 K 和 V？

因为 K 和 V **只依赖于当前位置的输入**，而不会因为后续生成了新 token 而改变：

```
Step 0: 输入 ["The", "cat", "sat"] → K₀, V₀, K₁, V₁, K₂, V₂  ("cat" 的 K 和 V 已经算好了)
Step 1: 输入 [..., "on"]           → K₃, V₃ (只需算新 token 的 K, V)
         Attention: Q₃ 和 [K₀,K₁,K₂,K₃] 做计算 → 需要历史 K,V！
Step 2: 输入 [..., "the"]          → K₄, V₄ (同上)
```

**没有 KV Cache**：每个 step 重新计算全部 K,V → 重复 O(n²) 次，总复杂度 O(n³)
**有 KV Cache**：每步只算新 token 的 K,V，从缓存读取历史 K,V → 总复杂度 O(n²)

### KV Cache 占多少显存？（关键计算）

这是理解推理框架设计的**核心数字**：

```python
# 一个 transformer block 的 KV Cache 大小
per_token_per_layer = 2          # K 和 V 各一份
                     * d_model   # 模型维度（如 4096 for LLaMA-7B）
                     * 2         # FP16, 每个元素 2 字节
# = 2 * 4096 * 2 = 16,384 字节 ≈ 16 KB

# 整个模型有 n_layers 层
per_token_total = per_token_per_layer * n_layers
# LLaMA-7B: 16KB * 32 layers = 512 KB per token

# 一个请求有 seq_len 个 token
per_request = per_token_total * seq_len
# 4096 token 的请求: 512KB * 4096 = 2 GB！

# N 个并发请求
total_kv_cache = per_request * N_concurrent
# 10 个并发请求: 2GB * 10 = 20 GB
```

**LLaMA-7B（FP16）的显存分解**：

| 项目 | 大小 | 占比 |
|------|------|------|
| 模型权重 | ~14 GB | ~58% |
| KV Cache (1 请求, 4096 tokens) | ~2 GB | ~8% |
| KV Cache (10 并发, 4096 tokens) | ~20 GB | — |
| 激活值 + 临时 buffer | ~2 GB | ~8% |

> **核心发现**：权重是固定的 ~14GB，但 KV Cache **随并发数和序列长度线性增长**。在 10 个并发 4096-token 请求时，KV Cache 可以远超模型权重本身！

这就是为什么：
- vLLM 发明了 **PagedAttention**（分块管理 KV Cache，减少碎片）
- llama.cpp 搞了 **KV Cache 量化**（把 K,V 从 FP16 压到 INT8/Q8_0）
- 各种 **prefix caching** 技术被提出（公共前缀的 KV Cache 多请求共享）

## 4. Prefill 和 Decode：两个阶段，两种瓶颈

Transformer 推理分为两个截然不同的阶段：

### Prefill 阶段（处理 prompt）

```
输入: "请解释相对论" (5 个 token)
操作: 一次性计算所有 5 个位置的 K, V
特性: compute-bound — 大量矩阵乘法，GPU 算力是瓶颈

Q, K, V 形状:
  Q: [5, d_model]  ← 多个 token 并行
  K: [5, d_model]  ← 全部存入 KV Cache
  V: [5, d_model]

Attention 计算:
  scores = Q @ K.T  ← [5, 5]，小矩阵，延迟可忽略
  output = scores @ V  ← [5, d_head]
```

### Decode 阶段（逐 token 生成）

```
输入: 只有新生成的那 1 个 token
操作: 只算这 1 个 token 的 Q, K, V
特性: memory-bound — 从显存读取巨大的 KV Cache，显存带宽是瓶颈

Q, K, V 形状:
  Q: [1, d_model]  ← 只有 1 个 token
  K_new: [1, d_model]  ← 新 token 的 K，追加到 KV Cache
  V_new: [1, d_model]

Attention 计算:
  scores = Q_new @ [K_cache, K_new].T  ← [1, seq_len] @ [seq_len, d_head].T
  output = scores @ [V_cache, V_new]   ← [1, d_head]

关键：K_cache 和 V_cache 可能很大（几万个 token），
从显存读它们的时间 >> 真正做矩阵乘法的时间
```

### 瓶颈转换

```
         Prefill 阶段          Decode 阶段
         ────────────          ───────────
GPU 算力   ████████████ (100%)    ██░░░░░░░░ (20%)
显存带宽   ██░░░░░░░░ (20%)      ████████████ (100%)

为什么 Decode 阶段算力利用率低？
→ GPU 的 Tensor Core 在"等"数据从 HBM 读进来
→ 就像高速公路（HBM 带宽）太窄，车（数据）堵在路上，
   工厂（Tensor Core）产能闲置
```

**这个瓶颈转换是理解所有推理优化的关键**：
- Continuous Batching 之所以有效，是因为 Decode 阶段算力闲置——多塞几个请求同时做，GPU 就有活干了
- FlashAttention 的本质是用 SRAM 做分块计算，减少 HBM 读写——直接打中 Decode 的 memory-bound 痛点

## 5. 一张图总结

```
┌──────────────────────────────────────────────────────────────┐
│                 Transformer 推理全景                           │
├──────────────────────┬───────────────────────────────────────┤
│     Prefill 阶段       │         Decode 阶段                    │
│   (处理 prompt)        │     (逐 token 生成)                    │
├──────────────────────┼───────────────────────────────────────┤
│ • 输入: 整个 prompt    │ • 输入: 1 个新 token                   │
│ • 并行算所有位置 K,V   │ • 只算新 token 的 K,V                  │
│ • Compute-bound       │ • Memory-bound                        │
│ • GPU 算力是瓶颈       │ • 显存带宽是瓶颈                        │
│ • 要 batch!           │ • 要 continuous batching!              │
├──────────────────────┴───────────────────────────────────────┤
│                    KV Cache                                   │
│  • 缓存所有历史 K,V，避免重复计算                               │
│  • 随序列长度 × 并发数线性增长                                  │
│  • 是推理显存的最大变量                                         │
│  • 管理 KV Cache = 管理推理服务的核心                           │
├───────────────────────────────────────────────────────────────┤
│  关键权衡:                                                     │
│  • 大 KV Cache → 更多并发，但每个请求可用的 KV Cache 更少        │
│  • 小 KV Cache → 每个请求可更长，但并发数降低                    │
│  • 量化 KV Cache → 省显存，但可能损失精度                       │
└───────────────────────────────────────────────────────────────┘
```

## 6. 下一步

现在你已经从推理视角理解了 Transformer 的核心机制：

- ✅ 训练并行 vs 推理串行 —— 这是所有差异的根源
- ✅ KV Cache 是什么、为什么需要、占多少显存
- ✅ Prefill 是 compute-bound，Decode 是 memory-bound
- ✅ 推理框架的所有优化本质上都在解决这三件事

接下来进入 **量化基础**，理解模型是如何从 FP16 压缩到 INT4 的——以及为什么 llama.cpp 的 Q4_K_M 和 TensorRT-LLM 的 FP8 是完全不同的思路。

---

## 动手实验：观察 KV Cache 增长

下面这段代码模拟了推理过程中 KV Cache 的大小变化——你可以调整参数感受一下。

In [1]:
# 模拟推理中 KV Cache 的显存占用

def kv_cache_size_gb(
    n_layers=32,        # Transformer 层数 (LLaMA-7B: 32)
    d_model=4096,       # 模型维度
    n_kv_heads=None,    # KV head 数量 (GQA: 可能 < n_heads)
    seq_len=4096,       # 序列长度 (prompt + 已生成的 tokens)
    n_concurrent=1,     # 并发请求数
    bytes_per_elem=2,   # FP16=2, FP8=1, INT8=1
):
    if n_kv_heads is None:
        n_kv_heads = d_model // 128  # 默认按 head_dim=128 算
    
    head_dim = d_model // (d_model // 128)  # 简化计算
    
    # 每个 token 的 KV Cache (所有层)
    per_token = 2 * n_layers * n_kv_heads * head_dim * bytes_per_elem
    
    # 总显存
    total = per_token * seq_len * n_concurrent
    return per_token, total / (1024**3)


print("=== KV Cache 显存占用模拟 ===\n")

# 不同模型规模
for name, layers, d_model in [
    ("LLaMA-7B  ", 32, 4096),
    ("LLaMA-13B ", 40, 5120),
    ("LLaMA-70B ", 80, 8192),
]:
    per_tok, total = kv_cache_size_gb(layers, d_model, seq_len=4096)
    print(f"{name}: per_token={per_tok/1024:.0f} KB, 1请求4K={total:.1f} GB, 10请求4K={total*10:.1f} GB")

print()

# 不同序列长度
for seq_len in [512, 1024, 2048, 4096, 8192, 32768]:
    _, total = kv_cache_size_gb(32, 4096, seq_len=seq_len, n_concurrent=8)
    print(f"LLaMA-7B, 8并发, seq_len={seq_len:5d}: KV Cache = {total:.1f} GB")

print()

# GQA 节省多少
print("=== GQA (Grouped Query Attention) 的效果 ===")
for name, kv_heads in [("MHA (1:1)  ", 32), ("GQA (8:1)  ", 8), ("GQA (4:1)  ", 4), ("MQA (1:1)  ", 1)]:
    _, total = kv_cache_size_gb(32, 4096, n_kv_heads=kv_heads, seq_len=4096, n_concurrent=8)
    saving = (1 - total / (kv_cache_size_gb(32, 4096, n_kv_heads=32, seq_len=4096, n_concurrent=8)[1])) * 100
    print(f"  {name}: KV Cache = {total:.1f} GB (节省 {saving:.0f}%)")

print()
print("关键洞察:")
print("  GQA 通过在多个 Q head 间共享 KV head，大幅减少 KV Cache")
print("  LLaMA-3 8B 使用 8 个 KV head (32 Q heads), KV Cache 只有 MHA 的 1/4")


=== KV Cache 显存占用模拟 ===

LLaMA-7B  : per_token=512 KB, 1请求4K=2.0 GB, 10请求4K=20.0 GB
LLaMA-13B : per_token=800 KB, 1请求4K=3.1 GB, 10请求4K=31.2 GB
LLaMA-70B : per_token=2560 KB, 1请求4K=10.0 GB, 10请求4K=100.0 GB

LLaMA-7B, 8并发, seq_len=  512: KV Cache = 2.0 GB
LLaMA-7B, 8并发, seq_len= 1024: KV Cache = 4.0 GB
LLaMA-7B, 8并发, seq_len= 2048: KV Cache = 8.0 GB
LLaMA-7B, 8并发, seq_len= 4096: KV Cache = 16.0 GB
LLaMA-7B, 8并发, seq_len= 8192: KV Cache = 32.0 GB
LLaMA-7B, 8并发, seq_len=32768: KV Cache = 128.0 GB

=== GQA (Grouped Query Attention) 的效果 ===
  MHA (1:1)  : KV Cache = 16.0 GB (节省 0%)
  GQA (8:1)  : KV Cache = 4.0 GB (节省 75%)
  GQA (4:1)  : KV Cache = 2.0 GB (节省 88%)
  MQA (1:1)  : KV Cache = 0.5 GB (节省 97%)

关键洞察:
  GQA 通过在多个 Q head 间共享 KV head，大幅减少 KV Cache
  LLaMA-3 8B 使用 8 个 KV head (32 Q heads), KV Cache 只有 MHA 的 1/4


## 补充资料

- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — Jay Alammar 的经典图解
- [LLM Inference Series (finbarr)](https://finbarr.ca/how-is-llama-cpp-possible/) — llama.cpp 的工作原理
- [vLLM PagedAttention 论文](https://arxiv.org/abs/2309.06180) — 理解 KV Cache 管理的 SOTA 方案
- [FlashAttention 论文](https://arxiv.org/abs/2205.14135) — 如何通过 IO-aware 算法加速 Attention
